In [63]:
import cv2
import numpy as np

def detect_and_match_features(img1, img2):
    # Create ORB detector (you can also use SIFT/SURF if available)
    orb = cv2.ORB_create()

    # Detect keypoints and compute descriptors
    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)

    # Use BFMatcher to match descriptors
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)

    # Sort matches based on distance
    matches = sorted(matches, key = lambda x:x.distance)

    return kp1, kp2, matches

def stitch_images(images):
    # Start with the first image as the base image
    base_image = images[0]

    for i in range(1, len(images)):
        # Find the keypoints and matches between the current base image and the next image
        kp1, kp2, matches = detect_and_match_features(base_image, images[i])
        
        # Extract the points from the keypoint matches
        points1 = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
        points2 = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
        print(points1)
        print(points2)
        # Find the homography matrix between the two sets of points
        M, _ = cv2.findHomography(points2, points1, cv2.RANSAC, 5.0)

        # Warp the next image to align it with the base image
        height, width = base_image.shape[:2]
        warped_image = cv2.warpPerspective(images[i], M, (width, height))

        # Blend the images together (you can use different blending strategies)
        base_image = blend_images(base_image, warped_image)
        
    return base_image

def blend_images(base_img, warped_img):
    # Simple linear blending (you could use other techniques like multi-band blending)
    result = cv2.addWeighted(base_img, 0.85, warped_img, 0.15, 0)
    return result

# Load the images you want to stitch (adjust the paths as needed)
img1 = cv2.imread('images/20250227_120628.JPG')
img2 = cv2.imread('images/20250227_120633.JPG')
img3 = cv2.imread('images/20250227_120637.jpg')
# img4 = cv2.imread('images/Capture4.jpg')
# img5 = cv2.imread('images/5.jpg')
# img6 = cv2.imread('images/6.jpg')

# Put all images in a list
images = [img1, img2, img3]
for img in images:
    img = cv2.resize(img, None, fx = .4, fy = .4, interpolation=cv2.INTER_LANCZOS4)
# Perform image stitching
stitched_image = stitch_images(images)
stitched_image = cv2.resize(stitched_image, None, fx = .3, fy = .3, interpolation=cv2.INTER_LANCZOS4)
# Show the result
cv2.imshow("Stitched Image", stitched_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


[[[2504.644   1207.5322 ]]

 [[3120.9514   971.0423 ]]

 [[3105.424    965.4684 ]]

 [[3564.0002  1417.2001 ]]

 [[2496.2832  1224.2538 ]]

 [[3183.6      938.4    ]]

 [[3120.7683   945.21606]]

 [[3091.3923   832.89606]]

 [[2493.297   1226.7421 ]]

 [[3093.4802   961.4871 ]]

 [[3120.7683   974.5921 ]]

 [[3375.8213   949.7089 ]]

 [[2493.8945  1225.4482 ]]

 [[3099.4524   963.8759 ]]

 [[3593.9314   963.8759 ]]

 [[3093.12     964.80005]]

 [[3106.9443   965.9521 ]]

 [[3092.9824   962.98004]]

 [[3375.6     1264.8    ]]

 [[3625.9202  1424.16   ]]

 [[3786.394    955.92975]]

 [[3122.8423   945.5618 ]]

 [[3185.0503   350.8532 ]]

 [[2040.4229   497.66412]]

 [[3105.424    955.51514]]

 [[3376.6511   950.53845]]

 [[3120.9514   945.96   ]]

 [[2086.8      490.80002]]

 [[3106.08     953.28   ]]

 [[3105.2163   953.8561 ]]

 [[3177.5854   350.8532 ]]

 [[3107.9124   358.31818]]

 [[3122.8423   358.31818]]

 [[2085.6963   489.02405]]

 [[2039.0402   497.66406]]

 [[3637.4404   953.8